In [ ]:
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
fig_dir="outputs/figures/for_slides"
os.makedirs(fig_dir, exist_ok=True)

files = glob.glob("outputs/runs/**/makeflowlog.csv", recursive=True)
paths = [os.path.dirname(p) for p in files]

In [ ]:
for num_cores in [4, 8, 16, 32, 64]:
    fig, ax = plt.subplots()
    for path in paths:
        if int(path.split("/")[3]) != 1:
            continue

        df = pd.read_csv(f'{path}/wq_log.csv')

        if df["num_cores"].iloc[0] != num_cores:
            continue
        if df["tasks_done"].iloc[-1] < 72:
            continue

        df.drop(df.index[-1], inplace=True)

        ax.plot(df["normalized_datetime"], df["workers_busy"], label=str(df.num_cores.iloc[0]))
        # ax.plot(df["normalized_datetime"], df["workers_connected"], label=str(df.num_cores.iloc[0]))
        # ax.plot(df["normalized_datetime"], df["workers_able"], label=str(df.num_cores.iloc[0]))

        ax.set_xlim(0, 35000)
        ax.set_ylim(0, 75)

        ax.set_xlabel("Time (s)", fontsize=14)
        ax.set_ylabel("Tasks Running", fontsize=14)
        ax.set_title(f"{num_cores} Core Workers", fontsize=16)
        ax.grid(True, linestyle="--", alpha=0.6)

    plt.savefig(f"{fig_dir}/workers vs. time {num_cores} cores.pdf", dpi=300, bbox_inches='tight')
    plt.savefig(f"{fig_dir}/workers vs. time {num_cores} cores.png", dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
fig, axes = plt.subplots(figsize=(10, 8))
# fig, axes = plt.subplots(figsize=(15, 8))
execution_times = pd.DataFrame(columns=["num_cores", "max_workers", "execution_time"])
cores_map = {
    4 : "red",
    8 : "blue",
    16: "green",
    32: "purple",
    64: "orange",
}

for path in paths:
    # must be first run of session.
    if int(path.split("/")[3]) != 1:
        continue
    df = pd.read_csv(f"{path}/wq_log.csv")
    sims_df = pd.read_csv(f"{path}/of_sims.csv")
    if df["tasks_done"].iloc[-1] < 72:
        continue

    df = df.iloc[:-1]  # drop last row

    
    max_workers = df[df["workers_connected"] == df["workers_connected"].max()].iloc[0]["normalized_datetime"]
    half_max_workers = df[df["workers_connected"] >= df["workers_connected"].max()/2].iloc[0]["normalized_datetime"]
    mean_exec_time = sims_df["execution_time"].mean()

    num_cores = df["num_cores"].iloc[0]
    new_row = pd.DataFrame({
        "num_cores": [num_cores],
        "max_workers": [max_workers],
        "execution_time": [mean_exec_time],
    })
    execution_times = pd.concat([execution_times, new_row], ignore_index=True)

    plt.scatter(half_max_workers, mean_exec_time, color=cores_map[num_cores], s=40)
    plt.scatter(max_workers, mean_exec_time,      color=cores_map[num_cores], s=90)
    plt.plot([half_max_workers, max_workers], [mean_exec_time, mean_exec_time], color=cores_map[num_cores], linestyle="dashed")
    # plt.plot([half_max_workers, max_workers], [mean_exec_time, mean_exec_time], color=cores_map[num_cores], linewidth=1.5, alpha=0.7)



plt.title(f"Wait Time vs. Execution Time (s)", fontsize=20)
plt.xlabel("Queuing Time (s)", fontsize=18)
plt.ylabel("Average Task Execution Time (s)", fontsize=18)
ax.tick_params(axis='both', labelsize=12)

plt.ylim(0)
plt.xlim(0)
axes.tick_params(axis="both", labelsize=14)
plt.grid(True, linestyle="--", alpha=0.6)

legend_handles = [
    Line2D(
        [0], [0],
        marker='o',
        linestyle='None',
        markerfacecolor=color,
        markeredgecolor=color,
        markersize=10,
        label=f"{cores}"
    )
    for cores, color in cores_map.items()
] + [
    Line2D([0], [0], marker='o', color='gray', linestyle='-', markersize=8,
           label="Half Max → Max Workers")
]

plt.legend(handles=legend_handles, title="CPU Cores")


plt.savefig(f"{fig_dir}/wait time vs. execution time.pdf", dpi=300, bbox_inches="tight")
plt.savefig(f"{fig_dir}/wait time vs. execution time.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
num_cores = 8

fig, axes = plt.subplots()
# fig, axes = plt.subplots(figsize=(15, 8))
for i in range(len(paths)):
    path = paths[i]
    if int(path.split("/")[3]) != 1:
        continue
    df = pd.read_csv(f"{path}/wq_log.csv")
    if df["num_cores"].iloc[0] != num_cores:
        continue
    if df["tasks_done"].iloc[-1] < 72:
        continue

    df = df.iloc[:-1]

    requested_cores = 72 * df["num_cores"].iloc[0]
    df["percent_cores_of_requested"] = (df["total_cores"]/requested_cores)*100
    df["percent_tasks_done"] = (df["tasks_done"]/74)*100
    df["percent_tasks_running"] = (df["tasks_running"]/74)*100

    plt.plot(df["normalized_datetime"], df["percent_tasks_running"],        label="Tasks Running")
    plt.plot(df["normalized_datetime"], df["percent_tasks_done"],           label="Tasks Completed")
    plt.plot(df["normalized_datetime"], df["percent_cores_of_requested"],   label="Cores Available")

    # Formatting each individual subplot
    plt.title(f"{num_cores} Core Workers", fontsize=16)
    plt.xlabel("Time (s)", fontsize=14)
    plt.ylabel("Percentage", fontsize=14)
    # plt.xlim(0, 33000)
    # plt.ylim(0, 110)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend(loc="best")
        
    plt.show()

In [ ]:
print(
    f"Core Count | Avg Exec Time (s) | Avg Queue Wait"
    " (s) | Allocation Granted | Total Workflow Time |"
    " Deadline Met?"
)

for num_cores in [4, 8, 16, 32, 64]:
    # Storage for metrics across all valid paths for the current core count
    sim_times = []
    wf_times = []
    max_alloc_times = []
    alloc_pcts = []
    in_time_flags = []

    for path in paths:
        if int(path.split("/")[3]) != 1:
            continue

        df = pd.read_csv(f"{path}/wq_log.csv")
        sims_df = pd.read_csv(f"{path}/of_sims.csv")

        if df["num_cores"].iloc[0] != num_cores:
            continue
        if df["tasks_done"].iloc[-1] < 72:
            continue

        df.drop(df.index[-1], inplace=True)

        max_allocation = (
            df["total_cores"].max() / (df["num_cores"].iloc[0] * 72)
        ) * 100
        max_workers = df[
            df["workers_connected"] == df["workers_connected"].max()
        ].iloc[0]["normalized_datetime"]
        mean_exec_time = sims_df["execution_time"].mean()
        exec_time = df["normalized_datetime"].iloc[-1]
        executed_in_time = exec_time < 7200

        # Append individual path metrics
        sim_times.append(mean_exec_time)
        wf_times.append(exec_time)
        max_alloc_times.append(max_workers)
        alloc_pcts.append(max_allocation)
        in_time_flags.append(executed_in_time)

    # Compute and display averages for this core count
    if sim_times:
        avg_sim = np.mean(sim_times)
        avg_wf = np.mean(wf_times)
        avg_alloc_time = np.mean(max_alloc_times)
        avg_alloc_pct = np.mean(alloc_pcts)
        pct_in_time = (
            np.mean(in_time_flags) * 100
        )  # Percentage of runs completed under 7200s

        print(
            f"{num_cores} | {avg_sim:.2f} | {avg_alloc_time:.2f} | {avg_wf:.2f}"
            f" | {avg_alloc_pct:.2f}% | {pct_in_time:.1f}%"
        )
    else:
        print(f"{num_cores} | No matching runs found")